In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Configuration
CHECKPOINT_DIR = "/content/drive/MyDrive/Colab Notebooks/checkpoints"
BATCH_SIZE = 64
EPOCHS = 100
LR = 0.001
IMG_SIZE = 48
EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Create checkpoint directory if not exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Custom Dataset
class FER2013Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        pixels = self.df.iloc[idx]['pixels']
        image = np.fromstring(pixels, dtype='uint8', sep=' ').reshape(IMG_SIZE, IMG_SIZE)
        image = image.astype(np.float32) / 255.0
        label = self.df.iloc[idx]['emotion']

        if self.transform:
            image = self.transform(image)
        else:
            image = torch.tensor(image).unsqueeze(0)

        return image, label



In [ ]:
import torch
import torch.nn as nn

class EmotionCNN(nn.Module):
    def __init__(self):
        super(EmotionCNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.5)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1)  # Flatten
        return self.classifier(x)

# Initialize the model and move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmotionCNN().to(device)


In [ ]:
# Preprocessing and Data Loading
def prepare_loaders(df= pd.read_csv('/content/drive/MyDrive/Colab Notebooks/fer2013_2.csv')):
    train_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomAffine(0, shear=10, scale=(0.8, 1.2)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.1)),
        transforms.RandomPerspective(distortion_scale=0.2),
        transforms.GaussianBlur(kernel_size=(3,3)),
        transforms.Normalize((0.5,), (0.5,))])

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))])

    train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['emotion'], random_state=42)

    train_dataset = FER2013Dataset(train_df, train_transform)
    test_dataset = FER2013Dataset(test_df, test_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, test_loader

train_loader, test_loader = prepare_loaders(pd.read_csv('/content/drive/MyDrive/Colab Notebooks/fer2013_2.csv'))

In [ ]:
# Training Function with Checkpoints
from torch.nn import CrossEntropyLoss
from sklearn.utils.class_weight import compute_class_weight

def train_model(model, train_loader, test_loader, optimizer, device):
    start_epoch = 0
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []

    # Calculate class weights for imbalance
    all_labels = []
    for _, labels in train_loader:
        all_labels.append(labels.cpu())
    all_labels = torch.cat(all_labels).numpy()
    class_weights = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # Modified criterion with label smoothing and class weights
    criterion = CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    # Initialize learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=3,
        verbose=True
    )
    # Checkpoint loading with scheduler state
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pth')
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        model.to(device)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        if EPOCHS > 40:
           scheduler.load_state_dict(checkpoint['scheduler_state'])
        start_epoch = checkpoint['epoch'] + 1
        train_losses = checkpoint['train_losses']
        test_losses = checkpoint['test_losses']
        train_accs = checkpoint['train_accs']
        test_accs = checkpoint['test_accs']
        print(f"Resuming training from epoch {start_epoch}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # Validation
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                test_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_loss = test_loss / len(test_loader)
        test_acc = 100 * correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)

        # Update learning rate
        scheduler.step(test_acc)

        # Save checkpoint with scheduler state
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'train_losses': train_losses,
            'test_losses': test_losses,
            'train_accs': train_accs,
            'test_accs': test_accs,
        }, checkpoint_path)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}% | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e}")

    # Save final model
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'final_model.pth'))
    return train_losses, test_losses, train_accs, test_accs


train_model(model=model.to(device), train_loader=train_loader, test_loader=test_loader, optimizer= optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5), device=device)


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Resuming training from epoch 34
Epoch [35/100] | Train Loss: 1.7734, Train Acc: 42.77% | Test Loss: 1.6774, Test Acc: 51.50% | LR: 2.50e-04
Epoch [36/100] | Train Loss: 1.7789, Train Acc: 42.51% | Test Loss: 1.6797, Test Acc: 52.73% | LR: 2.50e-04
Epoch [37/100] | Train Loss: 1.7760, Train Acc: 42.82% | Test Loss: 1.6796, Test Acc: 51.98% | LR: 2.50e-04
Epoch [38/100] | Train Loss: 1.7729, Train Acc: 43.24% | Test Loss: 1.6805, Test Acc: 52.63% | LR: 2.50e-04
Epoch [39/100] | Train Loss: 1.7742, Train Acc: 43.43% | Test Loss: 1.6814, Test Acc: 53.27% | LR: 2.50e-04
Epoch [40/100] | Train Loss: 1.7729, Train Acc: 43.44% | Test Loss: 1.6754, Test Acc: 52.48% | LR: 2.50e-04


In [ ]:
!pip install gradio
import gradio as gr
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

# Load model with error handling
model.eval()

# Emotion labels
emotion_labels = [
    "Angry 😠", "Disgust 🤢", "Fear 😨",
    "Happy 😄", "Sad 😞", "Surprise 😲",
    "Neutral 😐"
]

# Preprocessing with error prevention
preprocess = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

def predict_emotion(input_image):
    try:
        # Convert various input types to PIL Image
        if isinstance(input_image, np.ndarray):
            if input_image.shape[-1] == 4:  # RGBA image
                input_image = input_image[..., :3]  # Remove alpha channel
            pil_image = Image.fromarray(input_image.astype('uint8'), 'RGB')
        else:
            pil_image = input_image.convert('RGB')

        # Preprocess image
        input_tensor = preprocess(pil_image).unsqueeze(0)

        # Make prediction
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = torch.nn.functional.softmax(output[0], dim=0)

        # Create outputs
        confidences = {emotion_labels[i]: float(probabilities[i]) for i in range(7)}
        top_emotion = max(confidences, key=confidences.get)

        # Create visualization
        fig, ax = plt.subplots()
        ax.barh(list(confidences.keys()), list(confidences.values()))
        ax.set_xlim(0, 1)
        plt.close(fig)  # Prevent memory leaks

        return {top_emotion: confidences[top_emotion]}, fig

    except Exception as e:
        return f"Error: {str(e)}", None

# Create Gradio interface
demo = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Image(label="Upload Face Image", type="pil"),
    outputs=[
        gr.Label(label="Top Emotion Prediction"),
        gr.Plot(label="Confidence Distribution")
    ],
    title="Facial Emotion Recognition",
    description="Upload a face image to detect emotions (Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral)",
    examples=[
        ["example_images/happy.jpg"],
        ["example_images/sad.jpg"]
    ],
    allow_flagging="never"
)

if __name__ == "__main__":
    demo.launch(debug=True)

/usr/local/lib/python3.11/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated.Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0dbc67c27128c9a1cf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.11/dist-packages/gradio/processing_utils.py:158: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.savefig(output_bytes, format=fmt)
/usr/local/lib/python3.11/dist-packages/gradio/processing_utils.py:158: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.savefig(output_bytes, format=fmt)
/usr/local/lib/python3.11/dist-packages/gradio/processing_utils.py:158: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.savefig(output_bytes, format=fmt)
/usr/local/lib/python3.11/dist-packages/gradio/processing_utils.py:158: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.savefig(output_bytes, format=fmt)
/usr/local/lib/python3.11/dist-packages/gradio/processing_utils.py:158: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.savefig(output_bytes, format=fmt)
/usr/local/lib/python3.11/dist-pack